In [ ]:
````xml
<VSCode.Cell language="markdown">
# ML Experiment Tracking & Model Registry

**Chapter 9: AI Infrastructure**

This notebook demonstrates end-to-end experiment tracking and model lifecycle management for a BERT fine-tuning project.

**Running Example:**
- Task: Sentiment classification (IMDb reviews)
- Model: BERT-base fine-tuned
- Team: 3 members, 5 GPUs, 2 weeks of experiments
- Grid search: 3 learning rates × 2 batch sizes × 2 warmup steps = 12 runs

**Tools:**
- **MLflow**: Experiment tracking, model registry
- **DVC**: Data versioning

---

## Cell 1: Setup MLflow Tracking Server

Start a local MLflow tracking server to log experiments.

**Why MLflow?**
- Centralized experiment history
- Automatic metric visualization
- Model versioning and staging
- Reproducibility guarantees

**Setup:**
```bash
# Terminal 1: Start tracking server
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns
```

We'll connect to this server from our training code.

In [ ]:
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
from datetime import datetime
import json

# Configure MLflow tracking URI
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Create experiment for this project
experiment_name = "bert-sentiment-classification"
try:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        tags={
            "project": "sentiment-analysis",
            "team": "ml-infrastructure",
            "framework": "pytorch",
            "model_type": "bert-base"
        }
    )
except:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

print(f"✓ MLflow tracking server: http://127.0.0.1:5000")
print(f"✓ Experiment ID: {experiment_id}")
print(f"✓ Experiment name: {experiment_name}")

## Cell 2: Log First Experiment (Baseline)

Run baseline BERT fine-tuning and log everything to MLflow:
- **Hyperparameters**: learning_rate, batch_size, epochs, warmup_steps
- **Metrics**: accuracy, f1_score, loss (logged per epoch)
- **Artifacts**: model checkpoint, confusion matrix plot

**Tracking best practices:**
- Log system info (GPU, CUDA version)
- Tag runs with metadata (engineer name, branch)
- Save reproducible environment snapshot

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score
import io
from PIL import Image

def simulate_bert_training(learning_rate, batch_size, epochs, warmup_steps):
    """
    Simulate BERT fine-tuning for sentiment classification.
    In production, this would call transformers.Trainer or custom training loop.
    """
    metrics_history = []
    
    # Simulate training dynamics
    base_acc = 0.75
    base_f1 = 0.73
    lr_factor = 1.0 + (learning_rate - 3e-5) / 1e-4 * 0.05  # LR sensitivity
    batch_factor = 1.0 + (batch_size - 32) / 32 * 0.02
    
    for epoch in range(epochs):
        # Simulate metric progression
        progress = (epoch + 1) / epochs
        accuracy = min(0.95, base_acc * lr_factor * batch_factor + progress * 0.1 + np.random.normal(0, 0.01))
        f1 = min(0.94, base_f1 * lr_factor * batch_factor + progress * 0.12 + np.random.normal(0, 0.015))
        loss = max(0.1, 0.7 - progress * 0.5 + np.random.normal(0, 0.02))
        
        metrics_history.append({
            'epoch': epoch + 1,
            'accuracy': accuracy,
            'f1_score': f1,
            'loss': loss
        })
    
    return metrics_history

def create_confusion_matrix_plot():
    """Generate confusion matrix visualization."""
    # Simulate predictions for 1000 test samples
    y_true = np.random.choice([0, 1], size=1000, p=[0.5, 0.5])
    y_pred = y_true.copy()
    # Add some errors
    error_mask = np.random.choice([True, False], size=1000, p=[0.12, 0.88])
    y_pred[error_mask] = 1 - y_pred[error_mask]
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix - BERT Sentiment Classifier')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Save to bytes buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf

# Run baseline experiment
print("Running baseline BERT fine-tuning...")

with mlflow.start_run(run_name="baseline-bert") as run:
    # Log hyperparameters
    params = {
        'learning_rate': 5e-5,
        'batch_size': 32,
        'epochs': 3,
        'warmup_steps': 500,
        'optimizer': 'AdamW',
        'max_seq_length': 512,
        'model_name': 'bert-base-uncased'
    }
    mlflow.log_params(params)
    
    # Log system info as tags
    mlflow.set_tags({
        'engineer': 'alice',
        'gpu': 'A100-40GB',
        'cuda_version': '11.8',
        'branch': 'main',
        'training_date': datetime.now().strftime('%Y-%m-%d')
    })
    
    # Simulate training and log metrics per epoch
    metrics_history = simulate_bert_training(
        params['learning_rate'],
        params['batch_size'],
        params['epochs'],
        params['warmup_steps']
    )
    
    for metrics in metrics_history:
        mlflow.log_metrics({
            'accuracy': metrics['accuracy'],
            'f1_score': metrics['f1_score'],
            'loss': metrics['loss']
        }, step=metrics['epoch'])
        print(f"Epoch {metrics['epoch']}: acc={metrics['accuracy']:.4f}, f1={metrics['f1_score']:.4f}, loss={metrics['loss']:.4f}")
    
    # Log final test metrics
    final_metrics = metrics_history[-1]
    mlflow.log_metrics({
        'test_accuracy': final_metrics['accuracy'],
        'test_f1_score': final_metrics['f1_score']
    })
    
    # Create and log confusion matrix
    cm_buf = create_confusion_matrix_plot()
    mlflow.log_image(Image.open(cm_buf), "confusion_matrix.png")
    
    # Save and log model artifact (simulated)
    model_info = {
        'model_type': 'bert-base-uncased',
        'num_labels': 2,
        'final_accuracy': float(final_metrics['accuracy']),
        'trained_on': 'imdb-25k'
    }
    with open('model_info.json', 'w') as f:
        json.dump(model_info, f, indent=2)
    mlflow.log_artifact('model_info.json')
    
    run_id = run.info.run_id
    print(f"\n✓ Baseline run logged: {run_id}")
    print(f"✓ Final accuracy: {final_metrics['accuracy']:.4f}")
    print(f"✓ View in UI: http://127.0.0.1:5000/#/experiments/{experiment_id}")

## Cell 3: Hyperparameter Sweep (12 Experiments)

Run grid search over hyperparameter space:
- Learning rates: `[3e-5, 5e-5, 7e-5]`
- Batch sizes: `[16, 32]`
- Warmup steps: `[250, 500]`

**Total: 3 × 2 × 2 = 12 runs**

MLflow automatically tracks all runs, enabling:
- Parallel coordinates plots
- Metric comparison tables
- Best model selection by objective

**Real-world note:** This would run on multiple GPUs in parallel. Here we simulate serially for demonstration.

In [ ]:
from itertools import product
import time

# Define hyperparameter grid
learning_rates = [3e-5, 5e-5, 7e-5]
batch_sizes = [16, 32]
warmup_steps = [250, 500]

# Generate all combinations
param_grid = list(product(learning_rates, batch_sizes, warmup_steps))

print(f"Starting hyperparameter sweep: {len(param_grid)} runs\n")
print("Grid:")
print(f"  Learning rates: {learning_rates}")
print(f"  Batch sizes: {batch_sizes}")
print(f"  Warmup steps: {warmup_steps}")
print(f"\nRunning {len(param_grid)} experiments...\n")

sweep_results = []

for idx, (lr, bs, ws) in enumerate(param_grid, 1):
    with mlflow.start_run(run_name=f"sweep-run-{idx}") as run:
        # Log parameters
        params = {
            'learning_rate': lr,
            'batch_size': bs,
            'epochs': 3,
            'warmup_steps': ws,
            'optimizer': 'AdamW',
            'max_seq_length': 512,
            'model_name': 'bert-base-uncased'
        }
        mlflow.log_params(params)
        
        # Tag as sweep run
        mlflow.set_tags({
            'sweep': 'grid_search_v1',
            'run_number': idx,
            'engineer': np.random.choice(['alice', 'bob', 'charlie']),
            'gpu': np.random.choice(['A100-40GB', 'V100-32GB', 'RTX-3090'])
        })
        
        # Simulate training
        metrics_history = simulate_bert_training(lr, bs, 3, ws)
        
        # Log metrics per epoch
        for metrics in metrics_history:
            mlflow.log_metrics({
                'accuracy': metrics['accuracy'],
                'f1_score': metrics['f1_score'],
                'loss': metrics['loss']
            }, step=metrics['epoch'])
        
        # Log final test metrics
        final_metrics = metrics_history[-1]
        test_acc = final_metrics['accuracy']
        test_f1 = final_metrics['f1_score']
        
        mlflow.log_metrics({
            'test_accuracy': test_acc,
            'test_f1_score': test_f1
        })
        
        sweep_results.append({
            'run_id': run.info.run_id,
            'learning_rate': lr,
            'batch_size': bs,
            'warmup_steps': ws,
            'test_accuracy': test_acc,
            'test_f1_score': test_f1
        })
        
        print(f"Run {idx}/12: lr={lr:.0e}, bs={bs}, ws={ws} → acc={test_acc:.4f}, f1={test_f1:.4f}")

print(f"\n✓ Sweep complete: {len(sweep_results)} runs logged")
print(f"✓ View parallel coordinates plot in MLflow UI")
print(f"✓ URL: http://127.0.0.1:5000/#/experiments/{experiment_id}")

## Cell 4: MLflow UI Walkthrough

**Navigate to MLflow UI:** `http://127.0.0.1:5000`

**Key features demonstrated:**

1. **Run comparison table**
   - Sort by accuracy/F1 descending
   - Filter by tags (e.g., `engineer = "alice"`)
   - Select multiple runs → Compare

2. **Parallel coordinates plot**
   - Visualize hyperparameter impact
   - Drag axes to reorder
   - Brush to filter runs by metric range
   - Identify: high accuracy = high LR + small batch size

3. **Metric plots**
   - Plot accuracy vs. epoch for all runs
   - Overlay curves to compare convergence
   - Identify early stopping opportunities

4. **Artifact browser**
   - View confusion matrices side-by-side
   - Download model checkpoints
   - Inspect logged code versions

**Best run detection (visual):** Look for darkest line in parallel coords plot with high test_accuracy.

In [ ]:
# Programmatic UI walkthrough (display results in notebook)

# Convert sweep results to DataFrame
import pandas as pd

df_results = pd.DataFrame(sweep_results)

print("=== Hyperparameter Sweep Results ===\n")
print(df_results.sort_values('test_accuracy', ascending=False).to_string(index=False))

print("\n=== Top 3 Runs by Accuracy ===\n")
top3 = df_results.nlargest(3, 'test_accuracy')
for idx, row in top3.iterrows():
    print(f"Rank {idx+1}:")
    print(f"  Run ID: {row['run_id']}")
    print(f"  Learning rate: {row['learning_rate']:.0e}")
    print(f"  Batch size: {row['batch_size']}")
    print(f"  Warmup steps: {row['warmup_steps']}")
    print(f"  Test accuracy: {row['test_accuracy']:.4f}")
    print(f"  Test F1: {row['test_f1_score']:.4f}\n")

print("✓ Open MLflow UI to visualize parallel coordinates plot")
print("✓ Navigate to Experiments → bert-sentiment-classification")
print("✓ Select all runs → Click 'Compare' → Choose 'Parallel Coordinates'")

## Cell 5: Search API (Find Best Run)

Use MLflow Search API to programmatically find best runs by metric.

**Query syntax:**
- `metrics.test_accuracy > 0.85`
- `params.learning_rate = '5e-5'`
- `tags.engineer = 'alice'`

**Use cases:**
- Automatic model selection for deployment
- Hyperparameter analysis (which params correlate with high accuracy?)
- Team dashboards (who produced best models this sprint?)

In [ ]:
# Search for best run by test accuracy
from mlflow.entities import ViewType

# Query: Find runs with test_accuracy > 0.85, sort by accuracy descending
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="metrics.test_accuracy > 0.80",
    order_by=["metrics.test_accuracy DESC"],
    max_results=5,
    run_view_type=ViewType.ACTIVE_ONLY
)

print("=== Top 5 Runs by Test Accuracy ===\n")
print(runs[['run_id', 'params.learning_rate', 'params.batch_size', 'params.warmup_steps', 
            'metrics.test_accuracy', 'metrics.test_f1_score', 'tags.engineer']].to_string(index=False))

# Get best run
best_run = runs.iloc[0]
best_run_id = best_run['run_id']

print(f"\n=== Best Run ===")
print(f"Run ID: {best_run_id}")
print(f"Learning rate: {best_run['params.learning_rate']}")
print(f"Batch size: {best_run['params.batch_size']}")
print(f"Warmup steps: {best_run['params.warmup_steps']}")
print(f"Test accuracy: {best_run['metrics.test_accuracy']:.4f}")
print(f"Test F1: {best_run['metrics.test_f1_score']:.4f}")
print(f"Engineer: {best_run['tags.engineer']}")

print(f"\n✓ Best run identified: {best_run_id[:8]}...")
print("✓ Ready for model registry")

## Cell 6: Model Registry (Register Best Model)

Register the best model in MLflow Model Registry:
- **Purpose**: Centralized model lifecycle management
- **Stages**: None → Staging → Production → Archived
- **Benefits**: Version control, access control, audit trail

**Workflow:**
1. Register model from best run
2. Tag as "staging" for QA validation
3. Add description and metadata
4. Notify team for approval

**Model Registry vs. Experiment Tracking:**
- Experiments: All training runs (incl. failures)
- Registry: Production-worthy models only

In [ ]:
# Register best model in MLflow Model Registry

model_name = "bert-sentiment-classifier"

print(f"Registering model from run: {best_run_id[:8]}...\n")

# In production, you'd log the actual PyTorch model:
# mlflow.pytorch.log_model(model, "model")
# Here we simulate by registering the run's artifacts

# Create model version in registry
try:
    # Register model (creates new model if doesn't exist)
    model_version = mlflow.register_model(
        model_uri=f"runs:/{best_run_id}/model_info.json",
        name=model_name,
        tags={
            "task": "sentiment-classification",
            "framework": "pytorch",
            "base_model": "bert-base-uncased"
        }
    )
    
    version_number = model_version.version
    
    print(f"✓ Model registered: {model_name}")
    print(f"✓ Version: {version_number}")
    
except Exception as e:
    print(f"Note: {e}")
    print("Simulating model registration...")
    version_number = 1

# Transition to Staging stage
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Add description
client.update_model_version(
    name=model_name,
    version=version_number,
    description=f"""
    BERT sentiment classifier trained on IMDb-25k dataset.
    
    **Training details:**
    - Learning rate: {best_run['params.learning_rate']}
    - Batch size: {int(best_run['params.batch_size'])}
    - Test accuracy: {best_run['metrics.test_accuracy']:.4f}
    - Test F1: {best_run['metrics.test_f1_score']:.4f}
    
    **Trained by:** {best_run['tags.engineer']}
    **Run ID:** {best_run_id}
    """
)

# Transition to Staging
client.transition_model_version_stage(
    name=model_name,
    version=version_number,
    stage="Staging",
    archive_existing_versions=False
)

print(f"\n✓ Model transitioned to 'Staging' stage")
print(f"✓ Ready for QA validation")
print(f"✓ View in UI: http://127.0.0.1:5000/#/models/{model_name}")

## Cell 7: Load Model from Registry (Inference)

Load model from registry and run inference on test set.

**Registry benefits:**
- Load by stage: `models:/{model_name}/Staging`
- Load by version: `models:/{model_name}/{version}`
- Automatic lineage tracking (which run produced this model?)
- Deployment-ready artifact URI

**Production pattern:**
```python
# Serving code always loads from "Production" stage
model_uri = f"models:/{model_name}/Production"
model = mlflow.pytorch.load_model(model_uri)
```

In [ ]:
# Load model from Staging stage

model_uri = f"models:/{model_name}/Staging"

print(f"Loading model from registry...")
print(f"Model URI: {model_uri}\n")

# In production: model = mlflow.pytorch.load_model(model_uri)
# Here we simulate inference

# Get model metadata
model_version_details = client.get_model_version(
    name=model_name,
    version=version_number
)

print(f"=== Model Metadata ===")
print(f"Name: {model_version_details.name}")
print(f"Version: {model_version_details.version}")
print(f"Stage: {model_version_details.current_stage}")
print(f"Run ID: {model_version_details.run_id}")
print(f"Source: {model_version_details.source}")

# Simulate inference on test samples
test_samples = [
    "This movie was absolutely fantastic! Best film of the year.",
    "Terrible waste of time. Would not recommend.",
    "Average movie, nothing special but watchable.",
    "Brilliant performances and stunning cinematography!"
]

print(f"\n=== Inference Results ===")
print(f"Loaded model: {model_name} v{version_number} (Staging)\n")

for idx, text in enumerate(test_samples, 1):
    # Simulate prediction
    sentiment = "Positive" if np.random.rand() > 0.3 else "Negative"
    confidence = np.random.uniform(0.75, 0.98)
    
    print(f"Sample {idx}: {text[:50]}...")
    print(f"  Prediction: {sentiment} (confidence: {confidence:.2%})\n")

print("✓ Model loaded successfully from registry")
print("✓ Inference running on Staging model")

## Cell 8: Promote Model to Production

After QA validation passes, promote model from Staging → Production.

**Promotion workflow:**
1. QA team validates model on staging environment
2. Product owner approves metrics (accuracy, latency, fairness)
3. ML engineer promotes to Production stage
4. Archive previous production version

**Governance:**
- Audit trail: Who promoted when?
- Rollback capability: Revert to previous version
- Notifications: Alert team on production updates

In [ ]:
# Promote model to Production stage

print(f"Promoting {model_name} v{version_number} to Production...\n")

# Simulate QA approval
qa_checks = {
    'accuracy_threshold': best_run['metrics.test_accuracy'] > 0.85,
    'f1_threshold': best_run['metrics.test_f1_score'] > 0.83,
    'latency_p95': True,  # < 100ms
    'fairness_audit': True,  # Passed demographic parity check
}

print("=== Pre-Promotion Checks ===")
for check, passed in qa_checks.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{status}: {check}")

if all(qa_checks.values()):
    # Transition to Production
    client.transition_model_version_stage(
        name=model_name,
        version=version_number,
        stage="Production",
        archive_existing_versions=True  # Archive old Production versions
    )
    
    # Add promotion metadata
    client.set_model_version_tag(
        name=model_name,
        version=version_number,
        key="promoted_by",
        value="ml-engineer-alice"
    )
    
    client.set_model_version_tag(
        name=model_name,
        version=version_number,
        key="promotion_date",
        value=datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    )
    
    print(f"\n✓ Model promoted to Production")
    print(f"✓ Version {version_number} is now live")
    print(f"✓ Previous versions archived")
    print(f"\n✓ Production URI: models:/{model_name}/Production")
    print("✓ Update serving infrastructure to pull from Production stage")
else:
    print("\n✗ Promotion blocked: QA checks failed")

## Cell 9: DVC Setup (Version Training Dataset)

Use **DVC (Data Version Control)** to version large training datasets alongside code.

**Why DVC?**
- Git doesn't handle large files well (datasets, models)
- DVC stores data in remote storage (S3, GCS, Azure Blob)
- Lightweight Git commits + data snapshots = reproducibility

**Setup:**
```bash
# Initialize DVC
dvc init

# Add remote storage (e.g., S3 bucket)
dvc remote add -d myremote s3://ml-datasets-bucket/dvc-store

# Track dataset
dvc add data/imdb-25k.csv

# Commit DVC metadata (not data itself)
git add data/imdb-25k.csv.dvc data/.gitignore
git commit -m "Track IMDb dataset with DVC"

# Push data to remote
dvc push
```

**Result:** Dataset is versioned like code. Checkout old commit → `dvc pull` → get exact data.

In [ ]:
# Simulate DVC workflow (actual commands run in terminal)

import subprocess
import os

print("=== DVC Workflow Demo ===\n")

# In production, you'd run these commands:
dvc_commands = [
    "# Initialize DVC in repository",
    "dvc init",
    "",
    "# Configure remote storage (S3 example)",
    "dvc remote add -d myremote s3://ml-datasets-bucket/bert-sentiment",
    "dvc remote modify myremote region us-west-2",
    "",
    "# Track training dataset",
    "dvc add data/imdb_train.csv",
    "dvc add data/imdb_test.csv",
    "",
    "# Commit DVC metadata files to Git",
    "git add data/imdb_train.csv.dvc data/imdb_test.csv.dvc data/.gitignore",
    "git commit -m 'Track IMDb datasets with DVC v1.0'",
    "",
    "# Push data to remote storage",
    "dvc push",
    "",
    "# Later: Pull data on different machine",
    "git clone <repo_url>",
    "dvc pull  # Downloads data from S3",
]

for cmd in dvc_commands:
    print(cmd)

print("\n=== DVC + MLflow Integration ===\n")

# Log dataset version in MLflow
with mlflow.start_run(run_id=best_run_id):
    mlflow.set_tags({
        'dataset_version': 'imdb_train.csv:ab12cd34',  # DVC hash
        'dvc_remote': 's3://ml-datasets-bucket/bert-sentiment',
        'dataset_size': '25000 samples',
        'data_split': '20k train / 5k test'
    })

print("✓ Dataset versioned with DVC")
print("✓ DVC hash logged to MLflow run")
print("✓ Full reproducibility: code (Git) + data (DVC) + hyperparams (MLflow)")

print("\n=== Reproduce Experiment ===")
print("1. `git checkout <commit_hash>`")
print("2. `dvc pull` (gets exact dataset version)")
print("3. `mlflow run . -P run_id={best_run_id[:8]}` (reruns training)")

## Cell 10: Reproduce Experiment from Run ID

Full experiment reproduction workflow:
1. Fetch run metadata from MLflow
2. Retrieve code version (Git commit)
3. Pull dataset version (DVC)
4. Rerun training with exact hyperparameters

**Reproducibility guarantees:**
- **Code**: Git commit hash
- **Data**: DVC version hash
- **Hyperparameters**: MLflow logged params
- **Environment**: Conda env or Docker image

**Use cases:**
- Regulatory audits (FDA, financial services)
- Debugging model regressions
- Transfer learning from old experiments
- Paper submissions (reviewers can reproduce results)

In [ ]:
# Reproduce experiment from run ID

reproduce_run_id = best_run_id

print(f"=== Reproducing Experiment ===")
print(f"Run ID: {reproduce_run_id}\n")

# Step 1: Fetch run metadata
run = mlflow.get_run(reproduce_run_id)

print("--- Step 1: Fetch Metadata ---")
print(f"Experiment: {run.info.experiment_id}")
print(f"Start time: {datetime.fromtimestamp(run.info.start_time / 1000).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Status: {run.info.status}")

# Step 2: Extract hyperparameters
print("\n--- Step 2: Extract Hyperparameters ---")
params_to_reproduce = run.data.params
for key, value in params_to_reproduce.items():
    print(f"  {key}: {value}")

# Step 3: Get Git commit (from tags)
print("\n--- Step 3: Get Code Version ---")
git_commit = run.data.tags.get('mlflow.source.git.commit', 'HEAD')
git_branch = run.data.tags.get('branch', 'main')
print(f"  Git branch: {git_branch}")
print(f"  Git commit: {git_commit[:8]}...")

# Step 4: Get dataset version (from tags)
print("\n--- Step 4: Get Dataset Version ---")
dataset_version = run.data.tags.get('dataset_version', 'imdb_train.csv:ab12cd34')
dvc_remote = run.data.tags.get('dvc_remote', 's3://ml-datasets-bucket/bert-sentiment')
print(f"  Dataset: {dataset_version}")
print(f"  DVC remote: {dvc_remote}")

# Step 5: Reproduce command
print("\n--- Step 5: Reproduction Commands ---")
reproduction_script = f"""
# Checkout exact code version
git checkout {git_commit[:8]}

# Pull exact dataset version
dvc checkout data/imdb_train.csv
dvc pull

# Recreate Python environment
conda env create -f environment.yml
conda activate bert-sentiment

# Rerun training with exact hyperparameters
python train.py \\
  --learning_rate {params_to_reproduce['learning_rate']} \\
  --batch_size {params_to_reproduce['batch_size']} \\
  --epochs {params_to_reproduce['epochs']} \\
  --warmup_steps {params_to_reproduce['warmup_steps']} \\
  --mlflow_tracking_uri http://127.0.0.1:5000

# Verify results match original run
mlflow runs compare {reproduce_run_id[:8]} <new_run_id>
"""

print(reproduction_script)

print("\n✓ Reproduction workflow documented")
print("✓ All artifacts are version-controlled and traceable")

print("\n" + "="*60)
print("SUMMARY: ML Experiment Tracking & Model Registry")
print("="*60)
print(f"✓ Tracked {len(sweep_results)} experiments in MLflow")
print(f"✓ Best model: accuracy={best_run['metrics.test_accuracy']:.4f}, F1={best_run['metrics.test_f1_score']:.4f}")
print(f"✓ Model promoted to Production: {model_name} v{version_number}")
print(f"✓ Dataset versioned with DVC")
print(f"✓ Full reproducibility pipeline established")
print("\n📊 MLflow UI: http://127.0.0.1:5000")
print(f"🎯 Production model: models:/{model_name}/Production")